# 🌕 LUNA-DSS: Lunar Habitat AI & Site Suitability Engine
## Complete Data Cleaning, EDA, and Random Forest Machine Learning Pipeline

**Author:** Hackathon Project Team  
**Environment:** Google Colab / Local Jupyter Notebook  
**Dataset:** NASA LRO (LOLA, Diviner, LEND, CRaTER) Geospatial Lunar Measurements  

---
### 📌 Pipeline Overview
1. **Environment Setup & Library Imports**
2. **Dataset Acquisition & Colab Upload**
3. **Data Cleaning & Physical Constraint Validation**
4. **Exploratory Data Analysis (EDA) & Comprehensive Visualizations**
   - Feature distributions & skewness
   - Pearson correlation matrix heatmap
   - Spatial polar distribution maps
   - Radar spider chart for candidate landing sites
   - Interactive 3D scatter explorer (Plotly)
5. **Data Preprocessing & Stratified Train/Test Splitting**
6. **Model 1: Random Forest Regressor** (Continuous Suitability Score $0-100$)
7. **Model 2: Random Forest Classifier** (6-Zone Base Classification)
8. **Validation on 23 Official NASA/ISRO Lunar Benchmark Sites**
9. **Model Persistence (`joblib`) & Interactive Inference Engine**

--- 
## 🛠️ Step 1: Install & Import Libraries

In [ ]:
# Install required packages if running in Colab
!pip install -q scikit-learn pandas numpy matplotlib seaborn plotly joblib

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Set visual style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("✅ Libraries successfully imported!")

--- 
## 📂 Step 2: Load Dataset
If running in **Google Colab**, you can either upload the CSV directly via the upload cell or place it in a `data/` folder.

In [ ]:
# Check dataset path (supports both Colab and local directories)
possible_paths = [
    "data/lunar_ml_training_dataset.csv",
    "lunar_ml_training_dataset.csv",
    "../data/lunar_ml_training_dataset.csv"
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    print("⚠️ Dataset not found locally. If you are in Google Colab, upload it below:")
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Please make sure 'lunar_ml_training_dataset.csv' is present.")

df = pd.read_csv(data_path)
print(f"✅ Successfully loaded: {df.shape[0]:,} samples and {df.shape[1]} features from '{data_path}'")
df.head()

--- 
## 🧹 Step 3: Data Cleaning & Integrity Checks

In [ ]:
print("=" * 60)
print("DATA CLEANING & HEALTH REPORT")
print("=" * 60)

# 1. Missing / Null Values
null_counts = df.isnull().sum()
print(f"1. Null values count: {null_counts.sum()}")
if null_counts.sum() > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)
    print("   -> Imputed missing values with column medians.")

# 2. Duplicate Records
duplicates = df.duplicated().sum()
print(f"2. Duplicated rows count: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"   -> Removed {duplicates} duplicate rows.")

# 3. Physical Boundary & Sensor Validation (NASA/LRO Bounds)
initial_count = len(df)
df = df[
    (df["slope_deg"] >= 0.0) & (df["slope_deg"] <= 90.0) &
    (df["annual_illumination_pct"] >= 0.0) & (df["annual_illumination_pct"] <= 100.0) &
    (df["ice_prob"] >= 0.0) & (df["ice_prob"] <= 1.0) &
    (df["earth_vis_pct"] >= 0.0) & (df["earth_vis_pct"] <= 100.0) &
    (df["max_temp_k"] >= 20.0) & (df["min_temp_k"] >= 20.0) &
    (df["min_temp_k"] <= df["max_temp_k"])
]
print(f"3. Physical sanity filter: Removed {initial_count - len(df)} anomalies.")
print(f"   -> Clean dataset size: {len(df):,} valid samples.")

# Define 11 Core Geospatial Feature Names
feature_cols = [
    "slope_deg",
    "annual_illumination_pct",
    "ice_prob",
    "radiation_msv_yr",
    "earth_vis_pct",
    "elevation_m",
    "roughness_m",
    "max_temp_k",
    "min_temp_k",
    "shielding_factor",
    "weh_wt_pct"
]

df[feature_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]

--- 
## 📊 Step 4: Exploratory Data Analysis (EDA) & Visualizations

In [ ]:
# 1. Distribution of all 11 Lunar Features
plt.figure(figsize=(16, 11))
for i, col in enumerate(feature_cols, 1):
    plt.subplot(3, 4, i)
    sns.histplot(df[col], kde=True, color="royalblue", bins=30)
    plt.title(col, fontsize=11, fontweight="bold")
    plt.xlabel("")
plt.suptitle("1. Lunar Geospatial Feature Distributions (KDE & Histograms)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Target Distributions: Suitability Score (Continuous) & Zone Class (Categorical)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# A. Suitability Score
sns.histplot(df["suitability_score"], kde=True, color="teal", bins=35, ax=axes[0])
axes[0].axvline(df["suitability_score"].mean(), color="red", linestyle="--", label=f"Mean: {df['suitability_score'].mean():.1f}")
axes[0].set_title("A. Suitability Score Distribution (0 - 100)", fontsize=13, fontweight="bold")
axes[0].legend()

# B. Base Zone Class Balance
zone_counts = df["zone_class"].value_counts()
sns.barplot(x=zone_counts.values, y=zone_counts.index, palette="viridis", ax=axes[1])
axes[1].set_title("B. Zone Class Balance (Classification Target)", fontsize=13, fontweight="bold")
for i, count in enumerate(zone_counts.values):
    axes[1].text(count + 20, i, f"{count} ({count/len(df)*100:.1f}%)", va="center", fontweight="bold")

plt.suptitle("2. Target Variable Characteristics", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Pearson Correlation Heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df[feature_cols + ["suitability_score"]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("3. Feature Correlation Matrix (Pearson Correlation)", fontsize=14, fontweight="bold", pad=12)
plt.show()

In [ ]:
# 4. Multi-Constraint Trade-Off: Slope vs Illumination with Ice Probability
plt.figure(figsize=(12, 6))
scatter = plt.scatter(
    df["slope_deg"], df["annual_illumination_pct"],
    c=df["ice_prob"], cmap="coolwarm",
    s=df["suitability_score"] * 0.75, alpha=0.65, edgecolors="black", linewidths=0.2
)
plt.axvline(x=8.0, color="crimson", linestyle="--", label="Max Safe Habitat Slope (8°)")
plt.axhline(y=75.0, color="gold", linestyle="--", label="Min Solar Array Viability (75%)")

cbar = plt.colorbar(scatter)
cbar.set_label("Water Ice Probability", rotation=270, labelpad=15)
plt.title("4. Critical Multi-Objective Constraints: Slope vs Illumination (Bubble Size = Suitability)", fontsize=13, fontweight="bold")
plt.xlabel("Surface Slope (Degrees - Lower is Better)")
plt.ylabel("Annual Illumination (% - Higher is Better)")
plt.legend(loc="lower left", frameon=True)
plt.show()

In [ ]:
# 5. Multi-Criteria Radar / Spider Chart for Benchmark Candidate Sites
radar_categories = ['Illumination', 'Earth Line-of-Sight', 'Ice Deposits', 'Radiation Shielding', 'Trafficability (Low Slope)', 'Thermal Moderation']
N = len(radar_categories)

benchmark_profiles = {
    'Shackleton Rim (Habitat Candidate)': [91.5, 89.0, 75.0, 85.0, 92.0, 80.0],
    'Mons Malapert (Solar Power Peak)':   [89.0, 95.0, 20.0, 60.0, 94.0, 85.0],
    'Faustini Crater (Cryogenic Mining)': [10.0, 45.0, 95.0, 90.0, 65.0, 20.0]
}

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
plt.xticks(angles[:-1], radar_categories, color='navy', size=10, fontweight="bold")

for site_name, scores in benchmark_profiles.items():
    vals = scores + scores[:1]
    ax.plot(angles, vals, linewidth=2.5, linestyle='solid', label=site_name)
    ax.fill(angles, vals, alpha=0.15)

plt.title("5. Multi-Objective Radar Analysis of Benchmark Exploration Sites", size=13, fontweight="bold", y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.show()

In [ ]:
# 6. Interactive 3D Lunar Terrain & Suitability Visualization (Plotly)
sample_df = df.sample(min(2500, len(df)), random_state=42)

fig_3d = px.scatter_3d(
    sample_df,
    x="slope_deg",
    y="annual_illumination_pct",
    z="elevation_m",
    color="suitability_score",
    hover_name="zone_class",
    hover_data=["ice_prob", "radiation_msv_yr", "earth_vis_pct"],
    color_continuous_scale="Viridis",
    title="6. Interactive 3D Lunar Habitat Suitability Explorer (Rotate & Zoom)"
)
fig_3d.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig_3d.show()

--- 
## ⚙️ Step 5: Data Preprocessing & Train-Test Split

In [ ]:
# Extract Feature Matrix X and Target Vectors
X = df[feature_cols].values
y_reg = df["suitability_score"].values
y_cls = df["zone_class"].values

# 80% Train, 20% Test Split (Stratified by zone_class)
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.20, random_state=42, stratify=y_cls
)

print(f"[*] Training samples: {X_train.shape[0]:,} samples")
print(f"[*] Testing samples : {X_test.shape[0]:,} samples")

--- 
## 🌲 Step 6: Model 1 - Random Forest Regressor (Continuous Suitability Score)

In [ ]:
print("Training Random Forest Regressor...")
rf_reg = RandomForestRegressor(
    n_estimators=150,
    max_depth=14,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# 5-Fold Cross Validation
cv_scores = cross_val_score(rf_reg, X_train, y_reg_train, cv=5, scoring="r2", n_jobs=-1)
print(f"[*] 5-Fold CV R² Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Fit Model on Training Data
rf_reg.fit(X_train, y_reg_train)

# Evaluate on Unseen Test Set
y_reg_pred = rf_reg.predict(X_test)
r2 = r2_score(y_reg_test, y_reg_pred)
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("\n" + "=" * 50)
print(" REGRESSOR TEST EVALUATION METRICS")
print("=" * 50)
print(f"Test R² Score:              {r2:.4f} ({r2 * 100:.2f}%)")
print(f"Mean Absolute Error (MAE):  {mae:.3f} points")
print(f"Root Mean Sq Error (RMSE):  {rmse:.3f} points")

In [ ]:
# Model Diagnostics: Actual vs Predicted & Residual Error Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# A. Actual vs Predicted
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.35, color="teal", s=25)
axes[0].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], "r--", lw=2, label="Ideal Fit (y=x)")
axes[0].set_title(f"A. Actual vs Predicted Score (R² = {r2:.4f})", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Ground Truth Score")
axes[0].set_ylabel("Random Forest Predicted Score")
axes[0].legend()

# B. Residual Error (y_true - y_pred)
residuals = y_reg_test - y_reg_pred
sns.histplot(residuals, kde=True, color="coral", bins=30, ax=axes[1])
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title(f"B. Prediction Residual Errors (MAE = {mae:.2f} pts)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Residual Error")

plt.tight_layout()
plt.show()

--- 
## 🏷️ Step 7: Model 2 - Random Forest Classifier (Base Functional Zoning)

In [ ]:
print("Training Random Forest Classifier...")
rf_cls = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Fit Model
rf_cls.fit(X_train, y_cls_train)

# Evaluate on Test Set
y_cls_pred = rf_cls.predict(X_test)
accuracy = accuracy_score(y_cls_test, y_cls_pred)

print("\n" + "=" * 50)
print(" CLASSIFIER TEST EVALUATION METRICS")
print("=" * 50)
print(f"Overall Classification Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_cls_test, y_cls_pred, digits=4))

In [ ]:
# Confusion Matrix & Feature Importances
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# A. Confusion Matrix
cm = confusion_matrix(y_cls_test, y_cls_pred, labels=rf_cls.classes_)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=rf_cls.classes_, yticklabels=rf_cls.classes_, ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha="right")
axes[0].set_title("A. Classifier Confusion Matrix", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted Zone")
axes[0].set_ylabel("Actual Zone")

# B. Feature Importance Ranking
feat_imp = pd.Series(rf_reg.feature_importances_ * 100, index=feature_cols).sort_values(ascending=True)
feat_imp.plot(kind="barh", color="#1f77b4", ax=axes[1])
axes[1].set_title("B. Random Forest Feature Importance Ranking (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Relative Importance (%)")
for i, v in enumerate(feat_imp):
    axes[1].text(v + 0.3, i, f"{v:.1f}%", va="center", fontweight="bold")

plt.tight_layout()
plt.show()

--- 
## 🚀 Step 8: Validation on 23 Official NASA/ISRO Lunar Sites

In [ ]:
official_csv = "data/official_23_sites_ml_ready.csv"
if os.path.exists(official_csv):
    df_official = pd.read_csv(official_csv)
    print(f"Loaded {len(df_official)} Official Exploration Benchmark Sites!")
    
    X_off = df_official[feature_cols].values
    df_official["ai_predicted_score"] = np.round(rf_reg.predict(X_off), 1)
    df_official["ai_predicted_zone"] = rf_cls.predict(X_off)
    
    display_cols = ["node_id", "node_name", "slope_deg", "annual_illumination_pct", "ice_prob", "suitability_score", "ai_predicted_score", "ai_predicted_zone"]
    display(df_official[display_cols].head(10))
else:
    print("Official benchmark dataset not found in data/ folder. Skipping official site validation.")

--- 
## 💾 Step 9: Save Models & Interactive Inference Engine

In [ ]:
# 1. Save Trained Models using joblib
os.makedirs("models", exist_ok=True)
joblib.dump(rf_reg, "models/lunar_rf_regressor.joblib")
joblib.dump(rf_cls, "models/lunar_rf_classifier.joblib")
print("✅ Saved models to 'models/lunar_rf_regressor.joblib' and 'models/lunar_rf_classifier.joblib'")

# 2. Interactive Prediction Function for Any New Coordinates
def predict_lunar_site(features_dict):
    """
    Runs live inference on arbitrary input features.
    """
    input_df = pd.DataFrame([features_dict])[feature_cols]
    score = rf_reg.predict(input_df)[0]
    zone = rf_cls.predict(input_df)[0]
    probs = rf_cls.predict_proba(input_df)[0]
    
    print("=" * 55)
    print("🌕 AI LUNAR SITE ASSESSMENT REPORT")
    print("=" * 55)
    print(f"Predicted Suitability Score : {score:.2f} / 100")
    print(f"Operational Base Zone       : {zone}")
    print("\nZone Probability Breakdown:")
    for cls_name, p in zip(rf_cls.classes_, probs):
        bar = "█" * int(p * 25)
        print(f"  - {cls_name:30s}: {p*100:5.1f}% {bar}")

# Test Sample Input (Shackleton Crater Rim Alpha)
sample_input = {
    "slope_deg": 4.2,
    "annual_illumination_pct": 91.5,
    "ice_prob": 0.35,
    "radiation_msv_yr": 355.0,
    "earth_vis_pct": 89.0,
    "elevation_m": 1250.0,
    "roughness_m": 0.8,
    "max_temp_k": 220.0,
    "min_temp_k": 180.0,
    "shielding_factor": 0.22,
    "weh_wt_pct": 1.2
}

predict_lunar_site(sample_input)